# 

ZeroMQ: Request-Reply-Pattern

Ggfs. Requirements installieren.

In [1]:
!pip install pyzmq

Imports

In [2]:
import zmq
import threading
import time

Den Server definieren: Request-Reply-Pattern.
Also legen wir einen Reply Socket an.

In [3]:
def run_server():
    context = zmq.Context()
    socket = context.socket(zmq.REP)  # Reply-Socket

    socket.bind("tcp://*:12345")
    print("🟢 ZeroMQ Server läuft...")

    while True:
        message = socket.recv_string()  # blocking
        print("📥 Server empfängt:", message)

        if "STOP" in message:
            print("🛑 Server wird beendet")
            break

        reply = message + " *"
        socket.send_string(reply)

    socket.close()
    context.term()

Client definieren. Request-Reply-Pattern, also wird der Client einen Request-Socket öffnen.

In [4]:
def run_client():
    context = zmq.Context()
    socket = context.socket(zmq.REQ)  # Request-Socket

    print("🔵 Client verbindet sich...")
    socket.connect("tcp://localhost:12345")

    # Nachricht senden
    msg = "Hello World"
    print("📤 Client sendet:", msg)
    socket.send_string(msg)
    print ("Nachricht gesendet.")
    
    # Antwort empfangen (REQ muss recv nach send machen!)
    reply = socket.recv_string()
    print("📥 Client empfängt:", reply)

    # STOP senden
    socket.send_string("STOP")
    
    socket.close()
    context.term()

Dieses Mal starten wir den Client vor dem Server!
Weil der Client aber blockiert, müssen wir ihn hier als Thread starten.

In [5]:
client_thread = threading.Thread(target=run_client, daemon=True)
client_thread.start()

# kurze Wartezeit, um das Verhalten zu demonstrieren, dass
# der Client mit dem Senden der Nachricht wartet, bis der
# Server läuft
time.sleep(1)

🔵 Client verbindet sich...
📤 Client sendet: Hello World
Nachricht gesendet.


Server startet später.
Hier können wir den Server non-threaded starten.

In [6]:
run_server()

🟢 ZeroMQ Server läuft...
📥 Server empfängt: Hello World
📥 Client empfängt: Hello World *
📥 Server empfängt: STOP
🛑 Server wird beendet


Beobachtetes Verhalten:
Obwohl der Client früher startet:
- `connect()` blockiert nicht
- ZeroMQ queued intern
- Sobald Server da ist, läuft die Kommunikation!

Bei REQ/REP blockiert der Client bei recv(), nicht bei send()!
Das bedeutet:
- Nachrichten werden „losgeschickt“
- aber Antwort kommt erst, wenn Server da ist